In [0]:
# ============================================================
# NOTEBOOK   : silver_products
# PURPOSE    : Clean and transform bronze.products table
# SOURCE     : fincompliance_catalog.bronze.products
# DESTINATION: fincompliance_catalog.silver.products
# TRANSFORMATIONS:
#   - Deduplicate on product_id
#   - Fix column name TYPO (lenght to length)
#   - Cast dimension columns to correct types
#   - Handle nulls in category and dimensions
#   - Join with category_translation table
#   - Add English category names
#   - Add product size classification
#   - Add weight classification
#   - Drop bronze audit columns
#   - Add silver_updated_at audit column
# ============================================================

In [0]:
%run ../configs/config

In [0]:
authenticate_spark(spark)
set_catalog(spark)

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    when,
    lit,
    current_timestamp,
    count,
    avg,
    round as spark_round
)
from pyspark.sql.types import (
    IntegerType,
    DoubleType
)

In [0]:
# ============================================================
# CELL 5 - READ FROM BRONZE
# ============================================================

df_products = spark.table(f"{BRONZE_DB}.products")

print("=" * 60)
print("BRONZE.PRODUCTS — Before Transformation")
print("=" * 60)
print(f"Total rows    : {df_products.count():,}")
print(f"Total columns : {len(df_products.columns)}")

print("\nColumn names and data types:")
for field in df_products.schema.fields:
    print(f"  {field.name:<45} : {field.dataType}")

print("\nSample data BEFORE transformation (3 rows):")
df_products.show(3, truncate=True)

print("\nNull counts per column:")
for c in df_products.columns:
    null_count = df_products \
        .filter(col(c).isNull()) \
        .count()
    if null_count == 0:
        print(f"  {c:<45} : {null_count:,} nulls")
    else:
        print(f"  {c:<45} : {null_count:,} nulls")

In [0]:
# ============================================================
# DEDUPLICATION
# Deduplicate on product_id
# product_id is the primary key
# ============================================================

rows_before = df_products.count()
print(f"Rows BEFORE deduplication : {rows_before:,}")

df_products_silver = df_products \
    .dropDuplicates(["product_id"])

rows_after = df_products_silver.count()
print(f"Rows AFTER deduplication  : {rows_after:,}")
print(f"Duplicates removed        : {rows_before - rows_after:,}")
print("Done - Deduplication complete")

In [0]:
# ============================================================
# FIX COLUMN NAME TYPOS
# product_name_lenght        → product_name_length
# product_description_lenght → product_description_length
# This is a real typo in the original Olist dataset
# withColumnRenamed fixes column names without
# changing the data inside them
# ============================================================

# Show columns BEFORE renaming
print("Columns BEFORE fixing typos:")
for col_name in df_products_silver.columns:
    print(f"  {col_name}")

# Fix typos using withColumnRenamed
df_products_silver = df_products_silver \
    .withColumnRenamed(
        "product_name_lenght",
        "product_name_length"
    ) \
    .withColumnRenamed(
        "product_description_lenght",
        "product_description_length"
    )

# Show columns AFTER renaming
print("\nColumns AFTER fixing typos:")
for col_name in df_products_silver.columns:
    print(f"  {col_name}")

print("\nDone - Column name typos fixed")

In [0]:
# HANDLE NULL CATEGORY NAMES
# 610 products have null category names
# Replace with "unknown" to keep the records
# ============================================================

# Count nulls BEFORE
nulls_before = df_products_silver \
    .filter(col("product_category_name").isNull()) \
    .count()
print(f"Null category names BEFORE : {nulls_before:,}")

# Replace null with unknown
df_products_silver = df_products_silver \
    .withColumn(
        "product_category_name",
        when(
            col("product_category_name").isNull(),
            lit("unknown")
        )
        .otherwise(col("product_category_name"))
    )

# Count nulls AFTER
nulls_after = df_products_silver \
    .filter(col("product_category_name").isNull()) \
    .count()
print(f"Null category names AFTER  : {nulls_after:,}")
print(f"Nulls handled              : {nulls_before - nulls_after:,}")
print("Done - Null category names handled")

In [0]:
# HANDLE NULL DIMENSION COLUMNS
# product_weight_g  : 2 nulls
# product_length_cm : 2 nulls
# product_height_cm : 2 nulls
# product_width_cm  : 2 nulls
# Replace with 0 for numeric columns
# Cannot use "unknown" for numeric columns
# ============================================================

# Count nulls BEFORE
print("Null counts BEFORE handling:")
dimension_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]
for c in dimension_cols:
    null_count = df_products_silver \
        .filter(col(c).isNull()) \
        .count()
    print(f"  {c:<25} : {null_count:,} nulls")

# Replace nulls with 0
df_products_silver = df_products_silver \
    .withColumn(
        "product_weight_g",
        when(col("product_weight_g").isNull(), 0)
        .otherwise(col("product_weight_g"))
    ) \
    .withColumn(
        "product_length_cm",
        when(col("product_length_cm").isNull(), 0)
        .otherwise(col("product_length_cm"))
    ) \
    .withColumn(
        "product_height_cm",
        when(col("product_height_cm").isNull(), 0)
        .otherwise(col("product_height_cm"))
    ) \
    .withColumn(
        "product_width_cm",
        when(col("product_width_cm").isNull(), 0)
        .otherwise(col("product_width_cm"))
    )

# Count nulls AFTER
print("\nNull counts AFTER handling:")
for c in dimension_cols:
    null_count = df_products_silver \
        .filter(col(c).isNull()) \
        .count()
    print(f"  {c:<25} : {null_count:,} nulls")

print("Done - Null dimensions handled")

In [0]:
# ============================================================
# JOIN WITH CATEGORY TRANSLATION TABLE
# bronze.category_translation has:
# product_category_name         (Portuguese)
# product_category_name_english (English)
# This is the most important transformation
# in this notebook
# ============================================================

# Load category translation table
df_translation = spark.table(
    f"{BRONZE_DB}.category_translation"
)

print("Category translation table:")
print(f"Total rows : {df_translation.count():,}")
df_translation.show(5, truncate=False)

# Show categories BEFORE join
print("\nSample categories BEFORE join:")
df_products_silver \
    .select("product_category_name") \
    .distinct() \
    .orderBy("product_category_name") \
    .show(10, truncate=False)

# Perform LEFT JOIN
df_products_silver = df_products_silver \
    .join(
        df_translation.select(
            "product_category_name",
            "product_category_name_english"
        ),
        on="product_category_name",
        how="left"
    )

# Show categories AFTER join
print("\nSample categories AFTER join:")
df_products_silver \
    .select(
        "product_category_name",
        "product_category_name_english"
    ) \
    .distinct() \
    .orderBy("product_category_name") \
    .show(10, truncate=False)

# Check how many got English names
with_english = df_products_silver \
    .filter(
        col("product_category_name_english").isNotNull()
    ) \
    .count()
without_english = df_products_silver \
    .filter(
        col("product_category_name_english").isNull()
    ) \
    .count()

print(f"Products with English name    : {with_english:,}")
print(f"Products without English name : {without_english:,}")
print("Done - Category translation joined")

In [0]:
# ============================================================
# HANDLE NULL ENGLISH CATEGORY NAMES
# 623 products did not get English translation
# Replace null with "unknown"
# ============================================================

df_products_silver = df_products_silver \
    .withColumn(
        "product_category_name_english",
        when(
            col("product_category_name_english").isNull(),
            lit("unknown")
        )
        .otherwise(col("product_category_name_english"))
    )

# Verify no nulls remain
null_count = df_products_silver \
    .filter(
        col("product_category_name_english").isNull()
    ) \
    .count()
print(f"Null English names remaining : {null_count:,}")
print("Done - Null English names handled")

In [0]:
# ============================================================
# ADD PRODUCT SIZE CLASSIFICATION
# Classify products by their physical size
# using length x width x height
# Small  : volume < 1000 cm3
# Medium : volume 1000 to 10000 cm3
# Large  : volume > 10000 cm3
# This column DID NOT EXIST in bronze
# ============================================================

# Calculate volume first
df_products_silver = df_products_silver \
    .withColumn(
        "product_volume_cm3",
        col("product_length_cm") *
        col("product_height_cm") *
        col("product_width_cm")
    )

# Add size classification
df_products_silver = df_products_silver \
    .withColumn(
        "product_size",
        when(col("product_volume_cm3") == 0, "unknown")
        .when(col("product_volume_cm3") < 1000, "Small")
        .when(
            (col("product_volume_cm3") >= 1000) &
            (col("product_volume_cm3") <= 10000),
            "Medium"
        )
        .when(col("product_volume_cm3") > 10000, "Large")
        .otherwise("unknown")
    )

# Show size breakdown
print("Product size breakdown:")
df_products_silver \
    .groupBy("product_size") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print("Done - Product size classification added")

In [0]:
# ============================================================
# ADD WEIGHT CLASSIFICATION
# Classify products by weight
# Light  : weight < 500g
# Medium : weight 500g to 5000g
# Heavy  : weight > 5000g
# This column DID NOT EXIST in bronze
# ============================================================

df_products_silver = df_products_silver \
    .withColumn(
        "product_weight_class",
        when(col("product_weight_g") == 0, "unknown")
        .when(col("product_weight_g") < 500, "Light")
        .when(
            (col("product_weight_g") >= 500) &
            (col("product_weight_g") <= 5000),
            "Medium"
        )
        .when(col("product_weight_g") > 5000, "Heavy")
        .otherwise("unknown")
    )

# Show weight breakdown
print("Product weight breakdown:")
df_products_silver \
    .groupBy("product_weight_class") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# Show weight statistics
print("Weight statistics:")
df_products_silver \
    .filter(col("product_weight_g") > 0) \
    .select("product_weight_g") \
    .summary("min", "max", "mean") \
    .show()

print("Done - Weight classification added")

In [0]:
# ============================================================
# DROP BRONZE AUDIT COLUMNS
# ============================================================

df_products_silver = df_products_silver \
    .drop("ingestion_timestamp", "source_file")

print("Columns after dropping audit columns:")
for col_name in df_products_silver.columns:
    print(f"  {col_name}")
print(f"\nTotal columns : {len(df_products_silver.columns)}")
print("Done - Bronze audit columns dropped")

In [0]:
# ============================================================
# ADD SILVER AUDIT COLUMN
# ============================================================

df_products_silver = df_products_silver \
    .withColumn("silver_updated_at", current_timestamp())

print("Final columns in silver.products:")
for col_name in df_products_silver.columns:
    print(f"  {col_name}")
print(f"\nTotal columns : {len(df_products_silver.columns)}")
print(f"Total rows    : {df_products_silver.count():,}")
print("Done - Silver audit column added")

In [0]:
# ============================================================
# WRITE TO SILVER
# ============================================================

print("Writing to silver.products...")
print(f"Destination : {SILVER_DB}.products")
print(f"Total rows  : {df_products_silver.count():,}")
print(f"Total cols  : {len(df_products_silver.columns)}")

(
    df_products_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_DB}.products")
)

print(f"Written to {SILVER_DB}.products")

In [0]:
# VERIFICATION
# ============================================================

print("=" * 60)
print("SILVER.PRODUCTS VERIFICATION")
print("=" * 60)

df_silver = spark.table(f"{SILVER_DB}.products")

# CHECK 1 - Row count
bronze_count = spark.table(f"{BRONZE_DB}.products").count()
silver_count = df_silver.count()
print(f"\nCHECK 1 - Row counts:")
print(f"  Bronze : {bronze_count:,}")
print(f"  Silver : {silver_count:,}")
if bronze_count == silver_count:
    print(f"  Row counts match")
else:
    print(f"  Difference : {bronze_count - silver_count:,}")

# CHECK 2 - New columns
print(f"\nCHECK 2 - New columns:")
new_cols = [
    "product_name_length",
    "product_description_length",
    "product_category_name_english",
    "product_volume_cm3",
    "product_size",
    "product_weight_class",
    "silver_updated_at"
]
for c in new_cols:
    if c in df_silver.columns:
        print(f"  {c} exists")
    else:
        print(f"  {c} MISSING")

# CHECK 3 - Category translation
print(f"\nCHECK 3 - Category translation sample:")
df_silver \
    .select(
        "product_category_name",
        "product_category_name_english"
    ) \
    .distinct() \
    .orderBy("product_category_name") \
    .show(10, truncate=False)

# CHECK 4 - Size breakdown
print(f"CHECK 4 - Product size breakdown:")
df_silver \
    .groupBy("product_size") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# CHECK 5 - Weight breakdown
print(f"CHECK 5 - Product weight breakdown:")
df_silver \
    .groupBy("product_weight_class") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

# CHECK 6 - Null checks
print(f"CHECK 6 - Null checks:")
key_cols = [
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_size",
    "product_weight_class"
]
for c in key_cols:
    null_count = df_silver \
        .filter(col(c).isNull()) \
        .count()
    if null_count == 0:
        print(f"  {c:<35} : no nulls")
    else:
        print(f"  {c:<35} : {null_count:,} nulls")

print("=" * 60)
print("silver.products verification complete!")
print(f"Rows  : {silver_count:,}")
print(f"Cols  : {len(df_silver.columns)}")
print("=" * 60)